# NCBI Assembly Promote & Archive (Phase 3)

Promotes staged assembly files from S3 staging (written by CTS Phase 2)
to their final Lakehouse paths, archives replaced/suppressed assemblies,
and trims the transfer manifest for resumability.

Steps:
1. Configure staging prefix, removed manifest, updated manifest, and release tag
2. Scan staged files and display summary
3. Archive existing versions of updated assemblies (pre-overwrite)
4. Promote files to final paths with MD5 metadata
5. Archive replaced/suppressed assemblies
6. Trim manifest (remove promoted entries)

In [ ]:
"""Imports and S3 client initialisation."""

from __future__ import annotations

import json

from cdm_data_loaders.ncbi_ftp.promote import (
    DEFAULT_PATH_PREFIX,
    promote_from_s3,
)
from cdm_data_loaders.utils.s3 import get_s3_client

In [ ]:
"""Configure parameters."""

# S3 bucket where staged files and final Lakehouse data live
BUCKET = "cdm-lake"  # e.g. "cdm-lake"

# Staging prefix written by CTS Phase 2 (from the CTS output mount)
STAGING_PREFIX = "staging/run1/"  # e.g. "staging/run1/"

# Local path to removed_manifest.txt from Phase 1 (or None to skip archiving)
REMOVED_MANIFEST: str | None = None  # e.g. "output/removed_manifest.txt"

# Local path to updated_manifest.txt from Phase 1 (or None to skip pre-overwrite archiving)
UPDATED_MANIFEST: str | None = None  # e.g. "output/updated_manifest.txt"

# NCBI release tag for archive metadata (e.g. "2024-01")
NCBI_RELEASE: str | None = None

# S3 key of transfer_manifest.txt for trimming (or None to skip)
MANIFEST_S3_KEY: str | None = None  # e.g. "ncbi/transfer_manifest.txt"

# Final Lakehouse path prefix
PATH_PREFIX = DEFAULT_PATH_PREFIX

# Dry-run mode — log actions without making changes
DRY_RUN = True

print(f"Updated manifest: {UPDATED_MANIFEST}")
print(f"NCBI release:     {NCBI_RELEASE}")
print(f"Manifest S3 key:  {MANIFEST_S3_KEY}")
print(f"Path prefix:      {PATH_PREFIX}")

print(f"Dry-run:          {DRY_RUN}")
print(f"Path prefix:      {PATH_PREFIX}")

In [ ]:
"""Scan staged files and display summary."""

s3 = get_s3_client()
paginator = s3.get_paginator("list_objects_v2")

staged: list[str] = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=STAGING_PREFIX):
    staged.extend(obj["Key"] for obj in page.get("Contents", []))

sidecars = [k for k in staged if k.endswith((".md5", ".crc64nvme"))]
data_files = [k for k in staged if k not in set(sidecars)]

print(f"Staged objects: {len(staged)}")
print(f"  Data files:   {len(data_files)}")
print(f"  Sidecars:     {len(sidecars)}")

# Show first few data files
PREVIEW_COUNT = 10
for key in data_files[:PREVIEW_COUNT]:
    print(f"  {key}")
if len(data_files) > PREVIEW_COUNT:
    print(f"  ... and {len(data_files) - PREVIEW_COUNT} more")

In [ ]:
"""Promote staged files to final Lakehouse paths."""

report = promote_from_s3(
    staging_prefix=STAGING_PREFIX,
    bucket=BUCKET,
    removed_manifest=REMOVED_MANIFEST,
    updated_manifest=UPDATED_MANIFEST,
    ncbi_release=NCBI_RELEASE,
    manifest_path=MANIFEST_S3_KEY,
    path_prefix=PATH_PREFIX,
    dry_run=DRY_RUN,
)

print(json.dumps(report, indent=2))

In [ ]:
"""Display promotion report."""

print("=" * 50)
print("PROMOTION REPORT")
print("=" * 50)
print(f"Promoted:  {report['promoted']}")
print(f"Archived:  {report['archived']}")
print(f"Failed:    {report['failed']}")
print(f"Dry-run:   {report['dry_run']}")
print(f"Timestamp: {report['timestamp']}")

if report['failed'] > 0:
    print("\n⚠️  Some operations failed — check logs above for details.")

if report['dry_run']:
    print("\n📋 This was a dry-run. Set DRY_RUN = False and re-run to apply changes.")